# Estimate (CPU)

Everything after the reduction runs on CPU. The estimation tables land in about a
minute; the nulls (N1 at 2000 replicates) take the rest, and the full registered
run measured 54 minutes single-threaded on a laptop, so budget about an hour on a
Kaggle CPU session. Pass `--n-rep 500` for a 15-minute rehearsal. This notebook builds
the population, runs E1 to E4, and writes every table the paper cites.

Session settings: **CPU**, internet **on** for the install and the external check.

Attach the dataset holding the reduced runs from notebook 01, or point `RUNS` at
`/kaggle/working/runs` if this is the same session.

In [ ]:
!git clone -q https://github.com/garyzhang1006/seed-noise.git /kaggle/working/seed-noise
!pip -q install -e /kaggle/working/seed-noise

## Rehearse on a synthetic population first

The synthetic run has a truth you set, so it says whether the pipeline recovers a
known answer before it is pointed at data whose answer nobody knows.

In [ ]:
!seednoise analyze --synthetic --fast --rbar 0.15 \
    --out /kaggle/working/results-dry --n-boot 999

In [ ]:
import pandas as pd
dry = pd.read_csv("/kaggle/working/results-dry/tab_primary.csv")
dry[dry.label == "full contrast set"][["phenotype", "Lambda", "K_eff",
                                       "wild_lo", "wild_hi"]]

## The real population

`--n-boot 4999` is the registered setting. Gates that fail are printed at the end
of the run and each one names the fallback the paper takes in that case.

In [ ]:
RUNS = "/kaggle/input/datadecide-reduced/runs"   # or /kaggle/working/runs
OUT = "/kaggle/working/results"

In [ ]:
!seednoise analyze --runs {RUNS} --out {OUT} --n-boot 4999

## The headline

In [ ]:
primary = pd.read_csv(f"{OUT}/tab_primary.csv")
primary[primary.label.isin(["full contrast set", "batch-free contrast"])][
    ["phenotype", "label", "Lambda", "K_eff", "rbar_E", "wild_lo", "wild_hi"]]

## The mediators

`retained_excess` is the fraction of `Lambda - 1` that survives each control. A
column near one means the mediator explains little; near zero means it explains
almost everything.

In [ ]:
primary[primary.label.str.startswith("after")][
    ["phenotype", "label", "Lambda", "retained_excess"]]

## Gates and nulls

In [ ]:
gates = pd.read_csv(f"{OUT}/tab_gates.csv")
print(gates[["gate", "condition", "passed"]].to_string(index=False))
for _, g in gates[gates.passed == "no"].iterrows():
    print(f"\n{g.gate} failed -> {g.fallback}")

In [ ]:
nulls = pd.read_csv(f"{OUT}/tab_nulls.csv")
nulls.head(20)

## The external check

This costs nothing and reads an independent lab's replicate runs. Its `Lambda` is
attenuated by item noise that the released aggregates do not let us subtract, so
it is a floor rather than an estimate.

In [ ]:
!seednoise external --out {OUT}

In [ ]:
ext = pd.read_csv(f"{OUT}/tab_external.csv")
ext[["run_type", "metric", "partial_out", "R", "K", "rbar", "Lambda", "K_eff"]]

## What a practitioner does with it

In [ ]:
pd.read_csv(f"{OUT}/tab_practitioner.csv")